## Analyse der Standzeiten

In diesem Notebook werden die Standzeiten mit Geofence untersucht.

In [1]:
#Imports und Daten- Laden (EMR Daten mit 10 verschiedenen Formularbeschreibungen, bekommen aus dem abermaligen Run 
# des Cleaning Notebooks mit inkludierung zusätzlicher Formularbeschreibungen und hinzunehmen der Spalte Ort um Adresse zu bekommen)
import pandas as pd
import numpy as np
import re
from math import radians
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

df = pd.read_csv("IST_DATEN_MEHR_FORMULAR.csv", sep=",", parse_dates=["Meldungszeit"])
print(df.shape)
print(df["Formularbeschreibung"].value_counts().head(10))

(114756, 17)
Formularbeschreibung
Rollen           33828
Beginn Rollen    25198
Ende Rollen      25191
Anhalten         19244
Fahren            3704
Zündung aus       1735
Zündung an        1642
Stehen            1600
Abfahrt Kunde     1320
Ankunft Kunde     1294
Name: count, dtype: int64


Nun sehr viel mehr GPS Punkte inkludiert

In [2]:
#Hilfsfunktionen für Geofencing und Extraktion der PLZ

def haversine_vec(lat1, lon1, lat2_arr, lon2_arr):
    """Vektorisierte Haversine-Distanz in Metern."""
    R = 6371000
    dlat = np.radians(lat2_arr - lat1)
    dlon = np.radians(lon2_arr - lon1)
    a = (np.sin(dlat / 2) ** 2
         + np.cos(radians(lat1)) * np.cos(np.radians(lat2_arr))
         * np.sin(dlon / 2) ** 2)
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def extract_plz_city(pos):
    """PLZ und Stadtname aus 'DE - 12345 Stadt'-String extrahieren."""
    if pd.isna(pos):
        return None, None
    m = re.search(r"DE - (\d{5}) ([^,\(]+)", str(pos))
    if m:
        return m.group(1), re.sub(r"\(.*\)", "", m.group(2)).strip()
    return None, None


In [3]:
#GPS gruppieren
gps_grouped = {}
for (kz, dat), grp in df.groupby(["KENNZ_CLEAN", "DATUM"]):
    gps_grouped[(kz, dat)] = grp.sort_values("Meldungszeit").reset_index(drop=True)

print(f"\nGPS-Gruppen: {len(gps_grouped)}")


GPS-Gruppen: 206


In [4]:
#Formularbeschreibung Ankunft Kunde wird als der Punkt genommen, um den später der Geofence gezogen wird
ankunft_df = df[df["Formularbeschreibung"] == "Ankunft Kunde"].copy()
ankunft_df = ankunft_df[[
    "KENNZ_CLEAN", "TOURNR", "Meldungszeit",
    "Breitengrad", "Längengrad", "Position", "Fahrername", "DATUM"
]]
ankunft_df.rename(columns={
    "Meldungszeit": "Ankunft_Formular",
    "Breitengrad":  "lat_center",
    "Längengrad":   "lon_center"
}, inplace=True)

print(f"Ankunft-Events: {len(ankunft_df)}")

Ankunft-Events: 1294


Hauptsächliche Geofence funktion, hier wird ein Block gebildet wenn der LKW die Area betritt in der +-45 Minutenzeitspanne.
Wenn LKW Area verlässt wird Block beendet. Falls LKW den Bereich wieder betritt wird abermals ein Block gebildet (da LKW ja in der Zwischenzeit weg war).
Der längste Block wird ausgewählt weil das auch die längst Standzeit war. 
In nur circa 9 Fällen gab es zwei verschiedene Blocks in dem Zeitfenster, hier wurde der längste ausgewählt.

In [ ]:
def run_geofencing_v2(radius_m, gps_grouped, ankunft_df, window_min=45):
    WINDOW  = pd.Timedelta(f"{window_min}min")
    results = []

    for _, row in ankunft_df.iterrows():
        kz, dat      = row["KENNZ_CLEAN"], row["DATUM"]
        lat_c, lon_c = row["lat_center"], row["lon_center"]
        ankunft_t    = row["Ankunft_Formular"]
        key          = (kz, dat)

        base = {
            "KENNZ_CLEAN": kz, "TOURNR": row["TOURNR"],
            "Fahrername": row["Fahrername"], "DATUM": dat,
            "Position": row["Position"], "Ankunft_Formular": ankunft_t,
            "Eintritt_Geo": None, "Austritt_Geo": None,
            "Standzeit_Geo_min": None, "Punkte_im_Geofence": 0,
            "Radius_m": radius_m
        }

        if key not in gps_grouped:
            results.append(base); continue

        tdf  = gps_grouped[key]
        mask = ((tdf["Meldungszeit"] >= ankunft_t - WINDOW) &
                (tdf["Meldungszeit"] <= ankunft_t + WINDOW))
        win  = tdf[mask].copy()

        if len(win) == 0:
            results.append(base); continue

        win = win.copy()
        win["dist_m"] = haversine_vec(lat_c, lon_c,
                                      win["Breitengrad"].values,
                                      win["Längengrad"].values)
        win = win.sort_values("Meldungszeit").reset_index(drop=True)
        in_geo = win[win["dist_m"] <= radius_m].copy().reset_index(drop=True)

        if len(in_geo) == 0:
            results.append(base); continue

        if len(in_geo) == 1:
            base.update({
                "Eintritt_Geo": in_geo["Meldungszeit"].iloc[0],
                "Austritt_Geo": in_geo["Meldungszeit"].iloc[0],
                "Standzeit_Geo_min": 0,
                "Punkte_im_Geofence": 1
            })
            results.append(base); continue

        # Blocks nur trennen wenn Außenpunkte nachweisbar
        in_geo["block"] = 0
        current_block   = 0

        for i in range(1, len(in_geo)):
            t_prev = in_geo.loc[i-1, "Meldungszeit"]
            t_curr = in_geo.loc[i,   "Meldungszeit"]

            ausserhalb = win[
                (win["Meldungszeit"] > t_prev) &
                (win["Meldungszeit"] < t_curr) &
                (win["dist_m"] > radius_m)
            ]

            if len(ausserhalb) > 0:
                current_block += 1

            in_geo.loc[i, "block"] = current_block

        candidates = in_geo[in_geo["Meldungszeit"] >= ankunft_t - pd.Timedelta("5min")]

        if len(candidates) == 0:
            candidates = in_geo 

        # Größten Block wählen (meiste Punkte = längster tatsächlicher Aufenthalt)
        block_sizes = candidates.groupby("block").size()
        rel_block   = block_sizes.idxmax()

        block_df  = in_geo[in_geo["block"] == rel_block]
        eintritt  = block_df["Meldungszeit"].min()
        austritt  = block_df["Meldungszeit"].max()
        standzeit = round((austritt - eintritt).total_seconds() / 60, 1)

        base.update({
            "Eintritt_Geo": eintritt, "Austritt_Geo": austritt,
            "Standzeit_Geo_min": standzeit,
            "Punkte_im_Geofence": len(block_df)
        })
        results.append(base)

    result_df = pd.DataFrame(results)
    result_df["Qualitaet"] = "ok"
    result_df.loc[result_df["Punkte_im_Geofence"] == 1,
                  "Qualitaet"] = "unsicher (1 GPS-Punkt)"
    result_df.loc[result_df["Standzeit_Geo_min"].isna(),
                  "Qualitaet"] = "kein GPS-Treffer"
    return result_df

In [6]:
# Geofencing analyse
RADIUS    = 500  # Meter:  100 / 500 / 1000
WINDOW    = 45   # erst wurde größerer Rahmen von 2 Stunden probiert, aber da Fahrer manchmal zwischen zwei Orten hin und her 
                 #fahren macht es das geofencing schwierig und Zeitfenster wird verkleinert

geo_df = run_geofencing_v2(RADIUS, gps_grouped, ankunft_df,
                           window_min=WINDOW)

valid = geo_df[geo_df["Standzeit_Geo_min"].notna()].copy()
m     = valid["Standzeit_Geo_min"]

print(f"Stopps:        {len(geo_df)}")
print(f"Kein Treffer:  {geo_df['Standzeit_Geo_min'].isna().sum()}")
print(f"Ø Standzeit:   {m.mean():.1f} Min")
print(f"Median:        {m.median():.1f} Min")
print(f"0-Min:         {(m==0).sum()} ({(m==0).mean()*100:.1f}%)")
print(f"> 60 Min:      {(m>60).sum()}")
print(f"\n{geo_df['Qualitaet'].value_counts()}")

Stopps:        1294
Kein Treffer:  0
Ø Standzeit:   21.0 Min
Median:        16.4 Min
0-Min:         34 (2.6%)
> 60 Min:      15

Qualitaet
ok                        1260
unsicher (1 GPS-Punkt)      34
Name: count, dtype: int64


In [7]:
for r in [100, 500, 1000, 2000]:
    df_r = run_geofencing_v2(r, gps_grouped, ankunft_df)
    v    = df_r[df_r["Standzeit_Geo_min"].notna()]["Standzeit_Geo_min"]
    print(f"{r:>6}m | Ø {v.mean():.1f} | Median {v.median():.1f} | "
          f"0-Min {(v==0).mean()*100:.1f}% | >60: {(v>60).sum()}")

   100m | Ø 15.2 | Median 10.4 | 0-Min 2.6% | >60: 8
   500m | Ø 21.0 | Median 16.4 | 0-Min 2.6% | >60: 15
  1000m | Ø 23.9 | Median 20.4 | 0-Min 2.6% | >60: 19
  2000m | Ø 32.5 | Median 28.6 | 0-Min 2.6% | >60: 128


In [8]:
#Fahreranalyse
fahrer = (valid.groupby("Fahrername")["Standzeit_Geo_min"].agg(["mean", "median", "count", "max"]).round(1).reset_index())
fahrer.columns = ["Fahrername", "Avg", "Median", "Stopps", "Max"]
fahrer = fahrer.sort_values("Avg", ascending=False)
print(fahrer.to_string(index=False))
fahrer.to_csv('Ist_Fahrer.csv', index=False)

              Fahrername  Avg  Median  Stopps  Max
     Szmajdewicz, Marcin 38.5    42.6      76 75.0
         Teme, Abdoulaye 34.7    37.0      47 68.5
Calina, Florin Alexandru 33.2    39.8      84 68.9
        Hesse, Sven-Erik 27.0    28.8      36 66.0
       Krüger, Alexander 26.2    22.5     192 73.0
         Giewon, Mariusz 23.0    21.4     186 52.8
       Hrzuska, Krzystof 20.6    19.0      76 66.5
    Oelschlaeger, Marcel 18.6    16.0     127 82.5
      Jorczik, Christian 17.6    14.8     134 46.8
           Ledwina, Jens 10.1     8.7     166 47.1
           Borrs, Thomas  9.4     7.3     170 55.1


In [9]:
#Wochentagaanalyse
wt_map = {0:"Mo", 1:"Di", 2:"Mi", 3:"Do", 4:"Fr", 5:"Sa", 6:"So"}
valid["Wochentag"] = pd.to_datetime(valid["Ankunft_Formular"]).dt.dayofweek.map(wt_map)

wt = (valid.groupby("Wochentag")["Standzeit_Geo_min"].agg(["mean", "median", "count", "std"]).round(1).reset_index())
wt.columns = ["Wochentag", "Avg", "Median", "Stopps", "Std"]
wt["Wochentag"] = pd.Categorical(wt["Wochentag"],categories=["Mo","Di","Mi","Do","Fr","Sa","So"], ordered=True)
wt = wt.sort_values("Wochentag")
print(wt.to_string(index=False))

Wochentag  Avg  Median  Stopps  Std
       Mo 21.3    16.2     284 17.3
       Di 19.2    14.9     298 14.7
       Mi 21.6    17.2     280 16.1
       Do 21.5    17.0     292 16.6
       Fr 22.3    17.2     140 16.8


In [10]:
from scipy import stats

# Wochentag-Gruppen als separate Listen
wt_gruppen = {
    tag: grp["Standzeit_Geo_min"].values
    for tag, grp in valid.groupby("Wochentag")
}

#Kruskal-Wallis Test 
# Prüft ob mindestens ein Wochentag sich signifikant von den anderen unterscheidet
h_stat, p_kruskal = stats.kruskal(*wt_gruppen.values())

print("Kruskal Wallis Test")
print(f"H-Statistik: {h_stat:.3f}")
print(f"p-Wert:      {p_kruskal:.4f}")
print("nicht signifikant")

Kruskal Wallis Test
H-Statistik: 3.094
p-Wert:      0.5422
nicht signifikant


In [11]:
#Ortsanalyse mit mindestens 5 Stopps dort
valid[["PLZ","Stadt"]] = valid["Position"].apply(lambda x: pd.Series(extract_plz_city(x)))

orte = (valid.groupby(["PLZ","Stadt"])["Standzeit_Geo_min"].agg(["mean","median","count"]).round(1).reset_index())
orte.columns = ["PLZ", "Stadt", "Avg", "Median", "Stopps"]
orte = orte[orte["Stopps"] >= 5].sort_values("Avg", ascending=False)
print(orte.head(20).to_string(index=False))

  PLZ        Stadt  Avg  Median  Stopps
27374 Visselhövede 52.0    51.3      10
26180      Rastede 46.8    45.8       5
21683        Stade 37.7    41.5      44
25541  Brunsbüttel 37.1    38.8       5
23795 Bad Segeberg 35.2    40.8       5
22113      Hamburg 34.4    25.9       9
22844  Norderstedt 30.4    31.8      23
24589      Nortorf 28.1    21.2       9
20457      Hamburg 24.9    21.9     517
21107      Hamburg 23.9    21.0     124
22763      Hamburg 22.5    17.2       6
21682        Stade 19.5    23.0       7
20354      Hamburg 19.3    19.5       6
21029      Hamburg 19.1    16.1       7
22145        Braak 19.0    21.5       9
25524      Itzehoe 16.5    12.1      42
21493 Schwarzenbek 16.4    10.3       5
22419      Hamburg 14.3     3.3     119
22767      Hamburg 13.6    13.2      32
21337     Lüneburg 13.6     6.2       5


In [ ]:
#orte.to_csv('Ist_Ort.csv', index=False) das wird in dem Notebook für die Plots
#Standzeiten_Soll_Ist verwendet werden

In [13]:
# Visualisierungen
# Wochentag
fig = go.Figure()
fig.add_trace(go.Bar(
    x=wt["Wochentag"], y=wt["Avg"], name="Ø Standzeit",
    marker_color="#00498b",
    text=[f"{v:.1f}" for v in wt["Avg"]],
    textposition="outside", cliponaxis=False
))
fig.add_trace(go.Scatter(
    x=wt["Wochentag"], y=wt["Median"], name="Median",
    mode="lines+markers",
    line=dict(color="#ff0000", width=2.5, dash="dot"), marker=dict(size=9)
))
fig.update_layout(
    title=f"Standzeit nach Wochentag ({RADIUS}m Geofencing)",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Wochentag")
fig.update_yaxes(title_text="Standzeit (Min)", range=[0, 26])
fig.show()

# Fahrer
fah = fahrer.sort_values("Avg", ascending=True)
fig = go.Figure(go.Bar(
    y=fah["Fahrername"], x=fah["Avg"], orientation="h",
    marker_color="#00498b",
    text=[f"{v:.1f} min" for v in fah["Avg"]],
    textposition="outside", cliponaxis=False
))
fig.update_layout(title=f"Ø Standzeit pro Fahrer ({RADIUS}m Geofencing)")
fig.update_xaxes(title_text="Ø Standzeit (Min)", range=[0, 45])
fig.show()


In [14]:
# Top Städte
top = (valid.groupby("Stadt")["Standzeit_Geo_min"]
       .agg(["mean","count"]).reset_index()
       .query("count >= 5")
       .sort_values("mean", ascending=True).tail(10))
top.columns = ["Stadt","Avg","Stopps"]
fig = go.Figure(go.Bar(
    y=top["Stadt"], x=top["Avg"], orientation="h",
    marker_color="#00498b",
    text=[f"{v:.1f} min ({int(n)} Stopps)"
          for v, n in zip(top["Avg"], top["Stopps"])],
    textposition="outside", cliponaxis=False
))
fig.update_layout(title=f"Ø Standzeit nach Stadt ({RADIUS}m, mind. 5 Stopps)")
fig.update_xaxes(title_text="Ø Standzeit (Min)", range=[0, 61])
fig.show()


In [15]:
# Flop Städte
top = (valid.groupby("Stadt")["Standzeit_Geo_min"]
       .agg(["mean","count"]).reset_index()
       .query("count >= 5")
       .sort_values("mean", ascending=True).head(10))
top.columns = ["Stadt","Avg","Stopps"]
fig = go.Figure(go.Bar(
    y=top["Stadt"], x=top["Avg"], orientation="h",
    marker_color="#00498b",
    text=[f"{v:.1f} min ({int(n)} Stopps)"
          for v, n in zip(top["Avg"], top["Stopps"])],
    textposition="outside", cliponaxis=False
))
fig.update_layout(title=f"Ø Standzeit nach Stadt ({RADIUS}m, mind. 5 Stopps)")
fig.update_xaxes(title_text="Ø Standzeit (Min)", range=[0, 61])
fig.show()


Untersuchen der verschiedenen Geofence Radius Möglichkeiten

In [17]:
sens_data = []
for r in [100, 500, 1000,2000]:
    df_r = run_geofencing_v2(r, gps_grouped, ankunft_df)
    v    = df_r[df_r["Standzeit_Geo_min"].notna()]["Standzeit_Geo_min"]
    sens_data.append({
        "Label":    f"{r}m",
        "Avg":      round(v.mean(), 1),
        "Median":   round(v.median(), 1),
        "Zero_pct": round((v==0).mean()*100, 1),
        "Over60":   int((v>60).sum())
    })

sens_df = pd.DataFrame(sens_data)
labels  = sens_df["Label"].tolist()


In [19]:
print(sens_df.rename(columns={
    "Label":    "Radius",
    "Avg":      "Ø Standzeit",
    "Median":   "Median",
    "Zero_pct": "0-Min %",
    "Over60":   "> 60 Min"
}).to_string(index=False))

Radius  Ø Standzeit  Median  0-Min %  > 60 Min
  100m         15.2    10.4      2.6         8
  500m         21.0    16.4      2.6        15
 1000m         23.9    20.4      2.6        19
 2000m         32.5    28.6      2.6       128


Plot der verschiedenen Geofences

In [18]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=labels, y=sens_df["Avg"],
    mode="lines+markers+text", name="Ø Standzeit",
    line=dict(color="#00498b", width=3.5),
    marker=dict(size=13),
    text=[f"{v} Min" for v in sens_df["Avg"]],
    textposition="top center",
    textfont=dict(size=14, color="#00498b")
))

fig.add_trace(go.Scatter(
    x=labels, y=sens_df["Median"],
    mode="lines+markers+text", name="Median",
    line=dict(color="#ff0000", width=3.5, dash="dot"),
    marker=dict(size=13, symbol="diamond"),
    text=[f"{v} Min" for v in sens_df["Median"]],
    textposition="bottom center",
    textfont=dict(size=14, color="#ff0000")
))

fig.update_layout(
    title={"text": "Standzeit steigt mit Radius<br>"
           "<span style='font-size:17px;font-weight:normal'>"},
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5,
                font=dict(size=14)),
    yaxis=dict(range=[0, 39])
)
fig.update_xaxes(title_text="Geofencing-Radius", tickfont=dict(size=14))
fig.update_yaxes(title_text="Minuten", tickfont=dict(size=14))

fig.show()